# ASCII Maze SFT -> RL Walkthrough for Google Colab

This notebook adapts the repo's maze pipeline to a **Google Colab** workflow.
The repo's native trainers use **MLX**, which is Apple-Silicon-only, so this
notebook keeps the same maze generation, prompt format, reward function, and
measurement logic while swapping in a **PyTorch + LoRA** training stack that
runs on Colab GPUs.

The goal is to show the project lesson from `doc/plan.md` in a runnable form:

- **SFT** teaches the model to read the maze and output legal move sequences.
- **RL** then improves the policy with reward shaping, pushing solve rate higher.

By default the notebook stays focused on **3x3 mazes** so the signal appears in
one session. You can scale the dataset sizes and training steps later.

## What This Notebook Does

1. Installs a Colab-friendly training stack.
2. Reuses this repo's pure-Python maze code from `src/`.
3. Generates a compact 3x3 train/eval split.
4. Measures the base model.
5. Runs LoRA SFT on solved mazes.
6. Evaluates how much SFT improved conditioning and memorisation.
7. Runs a small **GRPO-style** RL loop on top of the SFT adapter.
8. Evaluates again to show how reward-driven training improves strategy.

The defaults are chosen for a Colab T4/A100 style runtime, not for maximum
final accuracy. If you want stronger results, increase the SFT and RL budgets.

In [ ]:
# Colab / notebook setup
# Pin pandas to the version Colab expects to avoid dependency churn in the runtime.
!pip -q install transformers accelerate peft datasets sentencepiece matplotlib "pandas==2.2.2"


In [ ]:
import math
import os
import random
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

DEFAULT_REPO_URL = "https://github.com/StephenJHardy/ascii-maze-rl.git"
DEFAULT_REPO_DIR = Path("/content/ascii-maze-rl")


def ensure_repo_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "src").exists():
        return cwd

    env_repo = os.environ.get("ASCII_MAZE_RL_REPO")
    if env_repo and (Path(env_repo) / "src").exists():
        return Path(env_repo)

    if DEFAULT_REPO_DIR.exists() and (DEFAULT_REPO_DIR / "src").exists():
        return DEFAULT_REPO_DIR

    print(f"Cloning repo into {DEFAULT_REPO_DIR} ...")
    subprocess.run(
        ["git", "clone", DEFAULT_REPO_URL, str(DEFAULT_REPO_DIR)],
        check=True,
    )
    return DEFAULT_REPO_DIR


REPO_ROOT = ensure_repo_root()
os.chdir(REPO_ROOT)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo_root={REPO_ROOT}")

from src.maze_dataset import MazeRecord
from src.maze_gen import generate
from src.maze_repr import SYSTEM_PROMPT, to_str
from src.maze_verify import extract_moves, manhattan_progress, reached_exit, simulate
from src.reward import compute_reward

device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"device={device} torch={torch.__version__}")


## Maze Task Refresher

The maze is rendered as a space-separated expanded grid. The model must output
only a move sequence such as `d r r d`.

The repo found two details were critical:

- the **chat template** matters, otherwise instruct models drift into explanations
- the reward must be **smooth** for partial solutions, not just solved / unsolved

The next cell shows a concrete maze and a few reward examples.

In [ ]:
example = generate(3, 3, seed=42)
print(to_str(example))
print()
print("optimal:", " ".join(example.solution_moves))

for completion in [
    "I think the answer is down right right down",
    "d",
    "d r r",
    "d r r d",
    "d r r d l l",
]:
    print(f"{completion!r:38s} -> reward={compute_reward(completion, example):.3f}")

## Experiment Config

These defaults are intentionally modest so the notebook remains practical on
Colab. The structure mirrors the repo's successful 3x3 experiment in the plan:
SFT first, then RL on top of the SFT adapter.

In [ ]:
@dataclass
class Config:
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"
    train_count: int = 512
    eval_count: int = 192
    train_seed_start: int = 0
    eval_seed_start: int = 10_000
    max_new_tokens: int = 24
    lora_rank: int = 16
    sft_epochs: int = 2
    sft_batch_size: int = 4
    sft_lr: float = 1e-4
    rl_steps: int = 150
    rl_group_size: int = 4
    rl_lr: float = 5e-6
    rl_beta: float = 0.05
    rl_temperature: float = 1.0
    eval_limit: int = 96
    output_root: str = "colab_runs"

CFG = Config()
OUTPUT_ROOT = Path(CFG.output_root)
OUTPUT_ROOT.mkdir(exist_ok=True)
SFT_DIR = OUTPUT_ROOT / "sft_adapter"
RL_DIR = OUTPUT_ROOT / "rl_adapter"
CFG

## Build a Compact Train / Eval Split

The training split uses one range of seeds; the evaluation split uses a disjoint
range. That keeps the held-out mazes structurally separate while preserving the
same task distribution.

In [ ]:
def make_records(count: int, seed_start: int, width: int = 3, height: int = 3):
    return [
        MazeRecord.from_maze(generate(width, height, seed=seed_start + i))
        for i in range(count)
    ]

train_records = make_records(CFG.train_count, CFG.train_seed_start)
eval_records = make_records(CFG.eval_count, CFG.eval_seed_start)

print(f"train={len(train_records)} eval={len(eval_records)}")
print("sample id:", train_records[0].id)
print(train_records[0].maze_str)
print("solution:", train_records[0].solution_moves)

In [ ]:
def set_seed(seed: int = 0):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(0)

## Shared Training API

From this point on, the notebook uses the repo's backend-agnostic API instead of
embedding trainer internals in notebook cells. The backend is chosen automatically:

- **MLX** on Apple Silicon when available
- **PyTorch + LoRA** on Colab / CUDA runtimes

That keeps the notebook focused on the experiment rather than framework details.

In [ ]:
from src.training import RLConfig, SFTConfig, evaluate_policy, load_policy, train_rl, train_sft

## Shared Evaluation Helpers

These are lightweight notebook helpers for turning repo evaluation objects into
DataFrames and displaying concrete successes and failures.

In [ ]:
def evaluate_to_frame(policy, records, title: str, limit: int | None = None, temperature: float = 0.0):
    subset = records if limit is None else records[:limit]
    results, summary = evaluate_policy(
        policy,
        subset,
        max_tokens=CFG.max_new_tokens,
        temperature=temperature,
        num_samples=1,
        verbose=False,
    )
    return pd.DataFrame([asdict(r) for r in results]), pd.Series({"name": title, **asdict(summary)})


def show_examples(df, records, n=5, sort_by="reward"):
    view = df.sort_values(sort_by).head(n)
    by_id = {r.id: r for r in records}
    for _, row in view.iterrows():
        record = by_id[row["maze_id"]]
        print("=" * 80)
        print(record.id)
        print(record.maze_str)
        print("target    :", record.solution_moves)
        print("completion:", row["completion"])
        print("reward    :", f"{row['reward']:.3f}")
        print("solved    :", row["solved"])

## Load the Base Model and Measure the Starting Point

This uses the same repo API the training stages will use later, so the notebook
stays backend-neutral.

In [ ]:
base_policy = load_policy(backend="auto", model=CFG.model_name)
base_eval_df, base_eval_summary = evaluate_to_frame(base_policy, eval_records, title="base_eval", limit=CFG.eval_limit)
pd.DataFrame([base_eval_summary])

## Run SFT

This call delegates the implementation details to the selected backend. On Colab,
it will use the PyTorch/PEFT path; on Apple Silicon it can use MLX.

In [ ]:
sft_output_dir = train_sft(
    SFTConfig(
        model=CFG.model_name,
        records=train_records,
        val_records=eval_records[:CFG.eval_limit],
        iters=CFG.train_count // CFG.sft_batch_size,
        epochs=CFG.sft_epochs,
        batch_size=CFG.sft_batch_size,
        lr=CFG.sft_lr,
        lora_rank=CFG.lora_rank,
        output_dir=SFT_DIR,
        backend="auto",
    )
)
sft_output_dir

## Evaluate After SFT

This is where we usually see the big jump in maze-conditioned behaviour.

In [ ]:
sft_policy = load_policy(backend="auto", model=CFG.model_name, adapter_path=sft_output_dir, lora_rank=CFG.lora_rank)
sft_train_df, sft_train_summary = evaluate_to_frame(sft_policy, train_records, title="sft_train", limit=CFG.eval_limit)
sft_eval_df, sft_eval_summary = evaluate_to_frame(sft_policy, eval_records, title="sft_eval", limit=CFG.eval_limit)

comparison = pd.DataFrame([
    base_eval_summary,
    sft_train_summary,
    sft_eval_summary,
])
comparison

## RL Stage

The RL stage now also lives in repo code. The notebook only passes a config and,
optionally, a custom reward function.

In [ ]:
# The default reward is src.reward.compute_reward.
# For a reward-design exercise notebook, replace reward_fn with a custom callable
# or with an import string like "my_rewards:shaped_reward".
reward_fn = None

rl_output_dir = train_rl(
    RLConfig(
        model=CFG.model_name,
        adapters=sft_output_dir,
        records=train_records,
        max_steps=CFG.rl_steps,
        num_generations=CFG.rl_group_size,
        max_tokens=CFG.max_new_tokens,
        temperature=CFG.rl_temperature,
        lr=CFG.rl_lr,
        beta=CFG.rl_beta,
        lora_rank=CFG.lora_rank,
        log_interval=10,
        save_interval=1000,
        output_dir=RL_DIR,
        backend="auto",
    ),
    reward=reward_fn,
)
rl_output_dir

## Evaluate After RL

This is the main comparison: base model, SFT checkpoint, then RL-on-top-of-SFT.

In [ ]:
rl_policy = load_policy(backend="auto", model=CFG.model_name, adapter_path=rl_output_dir, lora_rank=CFG.lora_rank)
rl_train_df, rl_train_summary = evaluate_to_frame(rl_policy, train_records, title="rl_train", limit=CFG.eval_limit)
rl_eval_df, rl_eval_summary = evaluate_to_frame(rl_policy, eval_records, title="rl_eval", limit=CFG.eval_limit)

final_comparison = pd.DataFrame([
    base_eval_summary,
    sft_train_summary,
    sft_eval_summary,
    rl_train_summary,
    rl_eval_summary,
])
final_comparison

In [ ]:
rl_log_path = Path(rl_output_dir) / "log.json"
rl_logs = pd.read_json(rl_log_path) if rl_log_path.exists() else pd.DataFrame()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if not rl_logs.empty:
    axes[0].plot(rl_logs["step"], rl_logs["reward_mean"], label="mean reward")
    axes[0].plot(rl_logs["step"], rl_logs["reward_max"], label="max reward")
    axes[0].legend()
axes[0].set_title("RL training rewards")
axes[0].set_xlabel("step")

bar_df = pd.DataFrame([
    {"stage": "base eval", "solve_rate": float(base_eval_summary["solve_rate"]), "mean_reward": float(base_eval_summary["mean_reward"])},
    {"stage": "SFT eval", "solve_rate": float(sft_eval_summary["solve_rate"]), "mean_reward": float(sft_eval_summary["mean_reward"])},
    {"stage": "RL eval", "solve_rate": float(rl_eval_summary["solve_rate"]), "mean_reward": float(rl_eval_summary["mean_reward"])},
])
bar_df.plot(x="stage", y=["solve_rate", "mean_reward"], kind="bar", ax=axes[1], rot=0)
axes[1].set_title("Held-out eval summary")
plt.tight_layout()
plt.show()

In [ ]:
show_examples(rl_eval_df, eval_records, n=5)

## Going Beyond the First 3x3 Result

The first 3x3 SFT -> RL run is useful as a **mechanism check**, but it is not the
strongest evidence in the project. The planning document later showed two more
important points:

- scaling **SFT** harder was the main driver of performance
- **RL helped most when aimed at a target size regime**, rather than as a universal fix

So the right way to read this notebook is:

1. the earlier 3x3 section proves the pipeline works end to end
2. the later runs show how the story becomes more convincing when the budget increases


In [ ]:
plan_results = pd.DataFrame([
    {"run": "3x3 SFT only", "train_setup": "1K mazes, 200 SFT iters", "size": "3x3", "solve_rate": 0.557},
    {"run": "3x3 SFT + GRPO", "train_setup": "same SFT + 500 RL steps", "size": "3x3", "solve_rate": 0.792},
    {"run": "Large SFT", "train_setup": "7.5K mazes, 2000 SFT iters", "size": "4x4", "solve_rate": 0.680},
    {"run": "Large SFT", "train_setup": "7.5K mazes, 5000 SFT iters", "size": "4x4", "solve_rate": 0.880},
    {"run": "Large SFT + targeted GRPO", "train_setup": "5K SFT + 500 RL steps on 4x4/5x5", "size": "4x4", "solve_rate": 0.900},
    {"run": "Longer targeted GRPO", "train_setup": "5K SFT + 1000 RL steps on 5x5-7x7", "size": "6x6", "solve_rate": 0.200},
])
plan_results


### Interpretation of the Later Results

Those later runs change the conclusion quite a bit. The convincing story is not
"RL alone solved mazes". It is:

- **SFT created the solver** by teaching maze-conditioned behaviour
- **more SFT budget kept helping** well past the first quick run
- **RL gave focused gains** when applied after a strong enough SFT checkpoint
- RL also introduced a real tradeoff: gains on the sizes being trained could come
  with regressions outside that distribution

That is the same pattern discussed in `doc/plan.md`: SFT is the main lever, RL is
best seen as a refinement stage rather than the whole solution.


## Optional Follow-On Colab Run

If you want a notebook section that goes beyond the minimal 3x3 proof, the next
best step is a **mixed-size curriculum**. The cell below sets up a larger train/eval
distribution that is still Colab-friendly. It is intentionally optional because it
takes longer and is more GPU-sensitive than the first run.


In [ ]:
FOLLOW_ON_SPECS = [
    {"width": 3, "height": 3, "count": 400, "seed_start": 20_000},
    {"width": 4, "height": 4, "count": 400, "seed_start": 30_000},
    {"width": 5, "height": 5, "count": 300, "seed_start": 40_000},
]

FOLLOW_ON_EVAL_SPECS = [
    {"width": 3, "height": 3, "count": 60, "seed_start": 120_000},
    {"width": 4, "height": 4, "count": 60, "seed_start": 130_000},
    {"width": 5, "height": 5, "count": 60, "seed_start": 140_000},
]

def make_records_from_specs(specs):
    records = []
    for spec in specs:
        records.extend(make_records(
            count=spec["count"],
            seed_start=spec["seed_start"],
            width=spec["width"],
            height=spec["height"],
        ))
    return records

follow_on_train_records = make_records_from_specs(FOLLOW_ON_SPECS)
follow_on_eval_records = make_records_from_specs(FOLLOW_ON_EVAL_SPECS)

pd.DataFrame([
    {"split": "train", "n": len(follow_on_train_records)},
    {"split": "eval", "n": len(follow_on_eval_records)},
])


### Recommended Follow-On Settings

For that larger mixed-size run, the next credible Colab target is:

- SFT on `follow_on_train_records` with a larger budget, for example `3-4` epochs
- RL on top of that checkpoint with a longer run such as `300-500` steps
- evaluate by size, not only overall average

The point of this follow-on section is to make the notebook match the planning
document more honestly: the interesting result is not just 3x3 imitation plus a bit
of RL, but that scaling SFT and then applying RL selectively gives the better story.


In [ ]:
def summarize_by_size(df, records):
    size_lookup = {r.id: f"{r.width}x{r.height}" for r in records}
    view = df.copy()
    view["size"] = view["maze_id"].map(size_lookup)
    return (
        view.groupby("size", as_index=False)
        .agg(
            n=("maze_id", "count"),
            solve_rate=("solved", "mean"),
            parse_rate=("parseable", "mean"),
            mean_reward=("reward", "mean"),
            mean_progress=("progress", "mean"),
        )
        .sort_values("size")
    )

# Example usage after running the larger experiment:
# follow_on_eval_df, follow_on_eval_summary = evaluate_model(policy_model, follow_on_eval_records, title="follow_on_eval")
# summarize_by_size(follow_on_eval_df, follow_on_eval_records)


## How to Read the Result

A typical pattern is:

- **Base model:** almost no maze solving.
- **After SFT:** parse rate jumps to nearly 100%, train performance rises fast,
  and held-out mazes improve because the model learned the task format and some
  local maze regularities.
- **After RL:** reward and solve rate improve again because the policy is being
  trained directly against path quality, not only imitation of demonstrations.

That matches the main repo finding from `doc/plan.md`:

- SFT is the stage that gives the model **maze-conditioned behaviour**.
- RL is the stage that turns that behaviour into a **better solver**.

## Ways to Scale This Notebook

If you want the notebook to track the stronger runs in the planning document,
these are the first levers to increase:

- raise `CFG.train_count` from `512` toward `1000+`
- raise `CFG.sft_epochs` or switch to a fixed-step SFT schedule
- raise `CFG.rl_steps` from `150` toward `500+`
- raise `CFG.rl_group_size` from `4` toward `8`
- add `4x4` mazes after the 3x3 pipeline is stable

For larger runs, save intermediate eval tables so you can see whether RL is
helping target sizes or regressing out-of-distribution sizes, which was one of
this repo's main lessons.